In [39]:
import pandas as pd


# Read the Excel file
file_path = 'Warehouse_Branch_LatLong_20250312.xlsx'
xls = pd.ExcelFile(file_path)

df = pd.read_excel(xls, sheet_name='Sheet1')
# Split LatLon column into separate latitude and longitude columns
df[['Latitude', 'Longitude']] = df['LatLon'].str.split(',', expand=True)

# Convert string values to float
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)
df = df.drop_duplicates(subset=['BranchNumber', 'LatLon'])
df

,cono,whse,divno,addr_1,addr_2,city,statecd,zipcd,BranchNumber,Branch_0,...,BranchName,Address,City2,StProv,ZipPostal,Country,Status,LatLon,Latitude,Longitude
0,1,02,2,5203 DIVISION AVE S,NaN,GRAND RAPIDS,MI,49548-5605,2,MI-002,...,Grand Rapids MI,5203 Division Avenue South,Grand Rapids,MI,49548-5605,US,Operating,"42.869134,-85.665481",42.869134,-85.665481
1,1,03,3,1325 INDUSTRY DR,NaN,TRAVERSE CITY,MI,49696-9245,3,MI-003,...,Traverse City MI,1325 Industry Drive,Traverse City,MI,49696-9245,US,Operating,"44.70974,-85.601561",44.709740,-85.601561
2,1,04,4,2121 HARVEY ST,NaN,MUSKEGON,MI,49442-6103,4,MI-004,...,Muskegon MI,2121 Harvey Street,Muskegon,MI,49442-6103,US,Operating,"43.214047,-86.206161",43.214047,-86.206161
3,1,05,5,11778 GREENWAY DR,NaN,HOLLAND,MI,49424-8654,5,MI-005,...,Holland MI,11778 Greenway Drive,Holland,MI,49424-8654,US,Operating,"42.806577,-86.071996",42.806577,-86.071996
4,1,06,6,3737 E MILHAM AVE,NaN,PORTAGE,MI,49002-9777,6,MI-006,...,Kalamazoo MI,3737 E Milham Avenue,Portage,MI,49002-9777,US,Operating,"42.231026,-85.544723",42.231026,-85.544723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337,1000,431,431,1602 TRICONT AVE,UNIT 5,WHITBY,ON,L1N 7C3,431,ON-431,...,Whitby ON,"1602 Tricont Avenue, Unit 5",Whitby,ON,L1N 7C3,CA,Operating,"43.86385,-78.900685",43.863850,-78.900685
338,1000,436,436,225 SPINNAKER WAY,UNIT 1,CONCORD,ON,L4K 5T8,436,ON-436,...,FBM Select Concord,225 Spinnaker Way,Concord,ON,L4K 5T8,CA,Operating,"43.827802,-79.492309",43.827802,-79.492309
339,1000,437,437,17260 HEATHER DR,UNIT 201,SURREY,BC,V3S 0B4,437,BC-437,...,BC Tool & Supply,17260 Heather Drive #201,Surrey,BC,V3S 0B4,CA,Operating,"49.105044,-122.743489",49.105044,-122.743489
340,1000,438,438,123 RESOURCE RD,NaN,KINGSTON,ON,K7P 0K1,438,ON-438,...,Kingston ON,"123 Resource Road, Units 2-7",Kingston,ON,K7P 0K1,CA,Operating,"44.272667,-76.562855",44.272667,-76.562855


In [40]:
df['BranchNumber'].value_counts()

BranchNumber
439    1
2      1
3      1
4      1
5      1
      ..
110    1
11     1
109    1
107    1
106    1
Name: count, Length: 318, dtype: int64

In [52]:
from google.cloud import bigquery
import pandas as pd

# Initialize a BigQuery client
client = bigquery.Client()

# Define the query
query = """
SELECT
  ProjectID,
  time_created,
  sourceFile,
  Addresses_Address,
  `Addresses_Address`[SAFE_OFFSET(0)].Longitude,
  `Addresses_Address`[SAFE_OFFSET(0)].Latitude,
  UpdateDate
FROM (
  SELECT
    ProjectID,
    time_created,
    sourceFile,
    Addresses_Address,
    `Addresses_Address`[SAFE_OFFSET(0)].Longitude,
    `Addresses_Address`[SAFE_OFFSET(0)].Latitude,
    UpdateDate,
    ROW_NUMBER() OVER (PARTITION BY ProjectID ORDER BY PARSE_DATE('%Y-%m-%d', UpdateDate) DESC) AS row_num
  FROM
    `proj-sales-recommender-dev`.`sales_recommender_dev`.`construct_connect_feed` 
  WHERE
     PARSE_DATE('%Y-%m-%d', UpdateDate) >= DATE_SUB(DATE '2025-01-16', INTERVAL 30 DAY)
    ) AS subquery
WHERE
  subquery.row_num = 1;
"""

# Execute the query
query_job = client.query(query)

# Convert the results to a pandas DataFrame
df_bq = query_job.to_dataframe()

# Display the DataFrame
df_bq

,ProjectID,time_created,sourceFile,Addresses_Address,Longitude,Latitude,UpdateDate
0,1007428290,2025-03-02 15:06:36.493053,History/1.4_DL_FBMSales_XML_20241229.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '385 E Van Buren St', 'AddressLine2': None, '...",-87.940580000,41.893914000,2024-12-28
1,1007446910,2025-03-02 15:06:36.493053,History/1.4_DL_FBMSales_XML_20241229.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '152 Enterprise Rd', 'AddressLine2': None, 'C...",-80.258118000,35.809988000,2024-12-28
2,1007163960,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '548-564 Cochrane Ave', 'AddressLine2': None,...",-122.887390000,49.252170000,2024-12-21
3,1007424499,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'Multiple Locations', 'AddressLine2': None, '...",-90.569937000,41.487481000,2024-12-28
4,1007447064,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '803 E Main St', 'AddressLine2': None, 'City'...",-95.618454000,34.232845000,2024-12-28
...,...,...,...,...,...,...,...
9495,1007453934,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'US-34', 'AddressLine2': None, 'City': 'Oneid...",-90.227297000,41.074033000,2025-01-16
9496,1007460146,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'To Be Determined', 'AddressLine2': None, 'Ci...",-88.319681000,41.666294000,2025-01-16
9497,1007463345,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '7922 N Dixie Dr', 'AddressLine2': None, 'Cit...",-84.203696000,39.827790000,2025-01-16
9498,1007418233,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'IL-62 & S New Wilke Rd', 'AddressLine2': Non...",-87.983562000,42.065315000,2025-01-16


In [53]:
import math

import numpy as np
import pandas as pd

def haversine_distance_miles_numpy(lat1, lon1, lat2_array, lon2_array):
    """
    Calculates the haversine distance between a single point (lat1, lon1)
    and an array of points (lat2_array, lon2_array) using NumPy for efficiency.

    Args:
        lat1 (float): Latitude of the single point.
        lon1 (float): Longitude of the single point.
        lat2_array (numpy.ndarray): Array of latitudes for the other points.
        lon2_array (numpy.ndarray): Array of longitudes for the other points.

    Returns:
        numpy.ndarray: Array of distances in miles.
    """
    R_miles = 6371.0 * 0.621371  # Earth's radius in miles

    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = np.radians(lat2_array)
    lon2_rad = np.radians(lon2_array)

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lon / 2) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    distances = R_miles * c
    return distances


def find_nearby_branch_haversine(row, df_branch, radius_miles=60, max_branches=5):
    """
    Find all branches within specified radius for each location in df_bq using haversine distance

    Args:
        df_bq: DataFrame with customer/location data
        df_branch: DataFrame with branch locations
        radius_miles: Search radius in miles
        max_branches: Maximum number of nearby branches to return

    Returns:
        pd.Series containing dictionaries of nearby warehouses and their distances
    """
    # Convert warehouse coordinates to numpy arrays for vectorized calculation
    branch_lats = df_branch["Latitude"].values
    branch_longs = df_branch["Longitude"].values

    distances = haversine_distance_miles_numpy(
        row["Latitude"], row["Longitude"], branch_lats, branch_longs
    )
    df_branch["distance"] = distances.round(2)
    nearby_pairs = df_branch[df_branch["distance"] <= radius_miles].sort_values(
        "distance"
    ).head(max_branches)
    nearby_pairs = nearby_pairs[
        ["BranchNumber", "distance", "Longitude", "Latitude", "City2", "StProv"]
    ].to_dict(orient="records")

    # Return np.nan if no nearby branches are found
    if not nearby_pairs:
        return np.nan

    return nearby_pairs


In [56]:
df_bq['branches'] = df_bq.apply(lambda x: find_nearby_branch_haversine(row=x, df_branch=df, radius_miles=60, max_branches=5), axis=1)
df_bq

,ProjectID,time_created,sourceFile,Addresses_Address,Longitude,Latitude,UpdateDate,branches
0,1007428290,2025-03-02 15:06:36.493053,History/1.4_DL_FBMSales_XML_20241229.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '385 E Van Buren St', 'AddressLine2': None, '...",-87.940580000,41.893914000,2024-12-28,"[{'BranchNumber': 129, 'distance': 6.09, 'Longitude': -88.051437, 'Latitude': 41.925029, 'City2'..."
1,1007446910,2025-03-02 15:06:36.493053,History/1.4_DL_FBMSales_XML_20241229.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '152 Enterprise Rd', 'AddressLine2': None, 'C...",-80.258118000,35.809988000,2024-12-28,"[{'BranchNumber': 289, 'distance': 26.09, 'Longitude': -79.92208, 'Latitude': 36.071819, 'City2'..."
2,1007163960,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '548-564 Cochrane Ave', 'AddressLine2': None,...",-122.887390000,49.252170000,2024-12-21,"[{'BranchNumber': 419, 'distance': 5.62, 'Longitude': -122.932609, 'Latitude': 49.176347, 'City2..."
3,1007424499,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'Multiple Locations', 'AddressLine2': None, '...",-90.569937000,41.487481000,2024-12-28,"[{'BranchNumber': 41, 'distance': 7.13, 'Longitude': -90.456207, 'Latitude': 41.545746, 'City2':..."
4,1007447064,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '803 E Main St', 'AddressLine2': None, 'City'...",-95.618454000,34.232845000,2024-12-28,NaN
...,...,...,...,...,...,...,...,...
9495,1007453934,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'US-34', 'AddressLine2': None, 'City': 'Oneid...",-90.227297000,41.074033000,2025-01-16,"[{'BranchNumber': 41, 'distance': 34.69, 'Longitude': -90.456207, 'Latitude': 41.545746, 'City2'..."
9496,1007460146,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'To Be Determined', 'AddressLine2': None, 'Ci...",-88.319681000,41.666294000,2025-01-16,"[{'BranchNumber': 129, 'distance': 22.59, 'Longitude': -88.051437, 'Latitude': 41.925029, 'City2..."
9497,1007463345,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '7922 N Dixie Dr', 'AddressLine2': None, 'Cit...",-84.203696000,39.827790000,2025-01-16,"[{'BranchNumber': 103, 'distance': 19.34, 'Longitude': -84.257757, 'Latitude': 39.550983, 'City2..."
9498,1007418233,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': 'IL-62 & S New Wilke Rd', 'AddressLine2': Non...",-87.983562000,42.065315000,2025-01-16,"[{'BranchNumber': 129, 'distance': 10.3, 'Longitude': -88.051437, 'Latitude': 41.925029, 'City2'..."


In [69]:
import pprint

pprint.pprint(df_bq['branches'].values[0])

[{'BranchNumber': 129,
  'City2': 'Addison',
  'Latitude': 41.925029,
  'Longitude': -88.051437,
  'StProv': 'IL',
  'distance': 6.09},
 {'BranchNumber': 49,
  'City2': 'Chicago',
  'Latitude': 41.818088,
  'Longitude': -87.656115,
  'StProv': 'IL',
  'distance': 15.55},
 {'BranchNumber': 47,
  'City2': 'Round Lake Park',
  'Latitude': 42.344522,
  'Longitude': -88.077919,
  'StProv': 'IL',
  'distance': 31.92},
 {'BranchNumber': 515,
  'City2': 'McHenry',
  'Latitude': 42.307459,
  'Longitude': -88.276188,
  'StProv': 'IL',
  'distance': 33.35},
 {'BranchNumber': 293,
  'City2': 'Gary',
  'Latitude': 41.594422,
  'Longitude': -87.371186,
  'StProv': 'IN',
  'distance': 35.91}]


In [71]:
n = 0
for idx, row in df_bq.iterrows():
    if str(row['branches']) == 'nan':
        print('No nearby branches found')
        print(row['branches'])
        continue
    for branch in row['branches']:
        # print(branch)
        project_dict ={'project_id': row['ProjectID'], 'time_created': row['time_created'].isoformat(), 'branch_number': branch['BranchNumber'], 'distance_from_branch': branch['distance']}
        print(project_dict)
    n +=1

{'project_id': 1007428290, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 129, 'distance_from_branch': 6.09}
{'project_id': 1007428290, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 49, 'distance_from_branch': 15.55}
{'project_id': 1007428290, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 47, 'distance_from_branch': 31.92}
{'project_id': 1007428290, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 515, 'distance_from_branch': 33.35}
{'project_id': 1007428290, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 293, 'distance_from_branch': 35.91}
{'project_id': 1007446910, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 289, 'distance_from_branch': 26.09}
{'project_id': 1007446910, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 259, 'distance_from_branch': 47.38}
{'project_id': 1007446910, 'time_created': '2025-03-02T15:06:36.493053', 'branch_number': 314, 'distance_from_branch': 52

In [73]:
df_bq.head(3)

,ProjectID,time_created,sourceFile,Addresses_Address,Longitude,Latitude,UpdateDate,branches
0,1007428290,2025-03-02 15:06:36.493053,History/1.4_DL_FBMSales_XML_20241229.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '385 E Van Buren St', 'AddressLine2': None, '...",-87.940580000,41.893914000,2024-12-28,"[{'BranchNumber': 129, 'distance': 6.09, 'Longitude': -88.051437, 'Latitude': 41.925029, 'City2'..."
1,1007446910,2025-03-02 15:06:36.493053,History/1.4_DL_FBMSales_XML_20241229.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '152 Enterprise Rd', 'AddressLine2': None, 'C...",-80.258118000,35.809988000,2024-12-28,"[{'BranchNumber': 289, 'distance': 26.09, 'Longitude': -79.92208, 'Latitude': 36.071819, 'City2'..."
2,1007163960,2025-03-03 09:41:18.493607,History/1.4_Adhoc_DL_FBMSales_XML_20250116.xml,"[{'ProjectAddressType': 'Project', 'AddressLine1': '548-564 Cochrane Ave', 'AddressLine2': None,...",-122.887390000,49.252170000,2024-12-21,"[{'BranchNumber': 419, 'distance': 5.62, 'Longitude': -122.932609, 'Latitude': 49.176347, 'City2..."


In [37]:
pd.options.display.max_colwidth = 100

df_bq[['Addresses_Address', 'branches', 'branch_count']].sort_values(by='branch_count', ascending=False).head(25).values

array([[array([{'ProjectAddressType': 'Project', 'AddressLine1': 'Multiple Locations', 'AddressLine2': None, 'City': 'Woodstock', 'CountryRegion': 'UNITED STATES', 'County': 'Mchenry', 'Latitude': Decimal('42.320586000'), 'Longitude': Decimal('-88.447623000'), 'StateProvince': 'IL', 'ZipPostalCode': '60098'}],
              dtype=object)                                                                                                                                                                                                                                                                                            ,
        list([{'BranchNumber': 515, 'distance': 8.81, 'Longitude': -88.276188, 'Latitude': 42.307459, 'City2': 'McHenry', 'StProv': 'IL'}, {'BranchNumber': 47, 'distance': 18.96, 'Longitude': -88.077919, 'Latitude': 42.344522, 'City2': 'Round Lake Park', 'StProv': 'IL'}, {'BranchNumber': 45, 'distance': 23.95, 'Longitude': -88.530698, 'Latitude': 42.661702, 'City2': 'Elkho

In [21]:
df[['BranchNumber', 'Address', 'City2', 'StProv']]

,BranchNumber,Address,City2,StProv
0,2,5203 Division Avenue South,Grand Rapids,MI
1,3,1325 Industry Drive,Traverse City,MI
2,4,2121 Harvey Street,Muskegon,MI
3,5,11778 Greenway Drive,Holland,MI
4,6,3737 E Milham Avenue,Portage,MI
...,...,...,...,...
337,431,"1602 Tricont Avenue, Unit 5",Whitby,ON
338,436,225 Spinnaker Way,Concord,ON
339,437,17260 Heather Drive #201,Surrey,BC
340,438,"123 Resource Road, Units 2-7",Kingston,ON


In [ ]:
def count_branches(row):
    if str(row) == 'nan':
        return 0
    return len(row)

radius_results = {}
for mile_radius in [30, 45, 60]:
    df_bq[f'whse_within_{mile_radius}miles'] = df_bq.apply(lambda x: find_nearby_branch_haversine(row=x, df_branch=df, radius_miles=mile_radius), axis=1)
    df_bq[f'whse_matching_count_{mile_radius}miles'] = df_bq[f'whse_within_{mile_radius}miles'].apply(count_branches)
    radius_results[f'{mile_radius}_total_included'] = len(df_bq[~df_bq[f'whse_within_{mile_radius}miles'].isna()])
    radius_results[f'{mile_radius}_percent_included'] = round(len(df_bq[~df_bq[f'whse_within_{mile_radius}miles'].isna()])/len(df_bq), 2)

    # Format mean, median, and max values
    mean_count = df_bq[f'whse_matching_count_{mile_radius}miles'].mean()
    median_count = df_bq[f'whse_matching_count_{mile_radius}miles'].median()
    max_count = df_bq[f'whse_matching_count_{mile_radius}miles'].max()
    
    radius_results[f'{mile_radius}_mean_whse_count'] = f"{mean_count:.2f}"
    radius_results[f'{mile_radius}_median_whse_count'] = f"{median_count:.2f}"
    radius_results[f'{mile_radius}_max_whse_count'] = f"{max_count:.2f}"

radius_results